In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('insurance.csv')
df

# EXPLORATORY DATA ANALYSIS

In [ ]:
df.shape


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
numeric_col = ['age','bmi', 'charges']
for i in  numeric_col:
    plt.figure(figsize=(6,4))
    sns.histplot( df[i] , kde = True, bins = 20)

In [ ]:
category_col = [ 'sex','smoker', 'region' , 'children']
for i in category_col:
    plt.figure(figsize=(6,4))
    sns.countplot( x = df[i] , color = 'orange' )

In [ ]:
for i in numeric_col:
    plt.figure(figsize=(6,4))
    sns.boxplot( x = df[i])

In [ ]:
"""
From the box plots we can conclude that there are many outliers present in charges section
a bit in bmi but in case of ages it is normally distributed !!!
"""

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df.corr(numeric_only = True) , annot = True)

In [ ]:
""" 
From the heatmap we can see the relation of age with other parameters
is very scarcitised , so we could actually think of removing it as a parameter
that we would be using for model prediction !!!
"""

# DATA CLEANING AND PROCESSING 

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned = df_cleaned.drop_duplicates()

In [ ]:
df_cleaned.shape

In [ ]:
# ONLY 1 ROW WAS DUPLICATE AS ONLY ONE ROW IS REMOVED

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
"""
While doing EDA you have to make sure that all the datatypes of the parameters are
in form of integers , floats , etc , not non-numeric as machine learning models 
can be implemented only on numeric datatypes.
"""

In [ ]:
df_cleaned.dtypes

In [ ]:
# HERE THERE ARE 3 PARAMETERS WHICH R NOT NUMERIC SO WE HAVE TO CONVERT THEM INTO NUMERIC 

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({"male" : 0 , "female" : 1})

In [ ]:
df_cleaned['sex'].head()

In [ ]:
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({"no" : 0 , "yes" : 1})

In [ ]:
df_cleaned.rename(
    columns = {
        "smoker" : "is_smoker",
        "sex"    : "is_female"
    } , inplace = True )

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies( df_cleaned , columns = ['region'] , drop_first = False)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = df_cleaned.astype(int)

# FEATURE ENGINEERING AND EXTRACTION

In [ ]:
sns.histplot(df_cleaned['bmi'] , bins = 20 )

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins = [0,18.5,24.5,29.9,float('inf')],
    labels = ['Underweight','Normal','Overweight','Obese'])
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies( df_cleaned , columns = ['bmi_category'] , drop_first = False)
df_cleaned.head()

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:
df_cleaned.head()

In [ ]:
print(df_cleaned.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
cols = ['age', 'bmi', 'children']
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])
df_cleaned.head()

In [ ]:
from scipy.stats import pearsonr

# Pearson Correlation Calculation ---->
# List of features to check against target
selected_features = [
    'age', 'bmi', 'children', 'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest','region_northeast',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese', 'bmi_category_Underweight'
]

correlations = {
    feature: pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation', ascending=False)

In [ ]:
cat_features = [
    'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
chi2_results = {}

for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by='p_value')
chi2_df

In [ ]:
final_df = df_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker', 'charges','region_southeast','bmi_category_Obese']]

In [ ]:
final_df.head()

In [ ]:
"""   ------------------------------------- END ---------------------------------------- """"